# STG-NF Shoplifting Detection — Google Colab

**Unsupervised shoplifting detection** using Spatio-Temporal Graph Normalizing Flows  
with the Periodic Adaptation Pipeline, trained on the RetailS dataset.

---

## Before you start

1. **Runtime → Change runtime type → GPU** (A100 if available via Colab Pro+)  
2. Upload **`RetailS.zip`** to the root of your Google Drive  
3. Run all cells top-to-bottom

---

## Hardware recommendations

| Runtime | VRAM | vs M4 MPS | Full training (60 epochs) | Verdict |
|---------|------|-----------|---------------------------|---------|
| **A100 40GB** | 40 GB | **~13×** | **~1.5 h** | ✅ Best choice |
| **L4** | 24 GB | ~7× | ~2.5 h | ✅ Great |
| V100 | 16 GB | ~8× | ~2 h | ✅ Good |
| T4 (free tier) | 16 GB | ~3× | ~5 h | ⚠️ Slow but works |
| **TPU v5-1** | — | — | — | ❌ **Not supported** |

> **Why no TPU?** TPU/XLA requires static graph compilation. This model has three blockers:
> (1) the `slogdet` call in `InvertibleLinear` is routed to CPU — XLA forbids host-device mixing;
> (2) the spatial coupling layers use Python `index_select` + `index_copy_` with runtime-computed
> indices — XLA cannot trace through dynamic shapes; (3) `BatchNorm` in ST-GCN blocks uses
> running statistics updated in-place — incompatible with XLA's functional model.
> Fixing all three would require a dedicated XLA rewrite. **Use A100 — zero code changes, 13× speedup.**


In [ ]:
# ── Cell 1: Hardware check ────────────────────────────────────────────────────
import subprocess, torch

try:
    print(subprocess.check_output('nvidia-smi --query-gpu=name,memory.total,driver_version '
                                   '--format=csv,noheader', shell=True).decode().strip())
except Exception:
    print('nvidia-smi not available')

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'Torch sees: {name}  ({vram:.1f} GB VRAM)')
    if   'A100' in name:            BATCH_SIZE, FLOW_DEPTH, NOTE = 1024, 12, 'A100'
    elif 'L4'   in name or 'A40' in name: BATCH_SIZE, FLOW_DEPTH, NOTE = 512,  12, 'L4/A40'
    elif 'V100' in name:            BATCH_SIZE, FLOW_DEPTH, NOTE = 512,  10, 'V100'
    else:                           BATCH_SIZE, FLOW_DEPTH, NOTE = 256,   8, 'T4/other'
    print(f'Auto-config: batch={BATCH_SIZE}  flow_depth={FLOW_DEPTH}  ({NOTE})')
else:
    BATCH_SIZE, FLOW_DEPTH = 128, 8
    print('WARNING: No CUDA GPU detected — go to Runtime > Change runtime type > GPU')


In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics', 'scipy', 'scikit-learn', 'tqdm'], check=True)
print('Dependencies ready.')


In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive'
RETAIL_ZIP = f'{DRIVE_ROOT}/RetailS.zip'

if not os.path.exists(RETAIL_ZIP):
    raise FileNotFoundError(
        f'RetailS.zip not found at {RETAIL_ZIP}.\n'
        'Upload it to the root of your Google Drive and re-run this cell.'
    )
print(f'RetailS.zip found ({os.path.getsize(RETAIL_ZIP)/1e6:.1f} MB)')


In [ ]:
import sys, json
from pathlib import Path

ROOT = Path('/content/STG_NF')
ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))

files = {
    "config.py": "\"\"\"\nCentralised hyperparameters and paths for the STG-NF system.\nAll other modules import from here \u2013 no magic numbers scattered around.\n\"\"\"\nfrom __future__ import annotations\nfrom pathlib import Path\n\n# \u2500\u2500 Paths \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nROOT          = Path(__file__).parent\nDATA_ROOT     = ROOT / \"data\"\nCHECKPOINT_DIR = ROOT / \"checkpoints\"\nBUFFER_DIR    = ROOT / \"buffer\"\n\nRETAIL_ZIP    = ROOT / \"RetailS.zip\"\nRETAIL_DIR    = DATA_ROOT / \"RetailS\"\n\nCHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)\nBUFFER_DIR.mkdir(parents=True, exist_ok=True)\n\n# \u2500\u2500 Skeleton / window \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nNUM_JOINTS    = 15          # 15-keypoint schema (no facial landmarks)\nCOORDS        = 2           # x, y only\nWINDOW_T      = 24          # frames per window\nSTRIDE        = 6           # sliding window stride\nINPUT_DIM     = NUM_JOINTS * COORDS   # 30 per frame\n\n# \u2500\u2500 Flow model \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nFLOW_DEPTH        = 8       # number of coupling layers\nHIDDEN_CHANNELS   = 64      # coupling-network hidden size\nACTNORM_INIT_ITERS = 256    # mini-batches for data-dependent ActNorm init\n\n# \u2500\u2500 Training \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nBATCH_SIZE    = 256\nLR            = 1e-4\nWEIGHT_DECAY  = 1e-5\nMAX_EPOCHS    = 100\nGRAD_CLIP     = 5.0\nMIX_RATIO     = 9           # 9 normal : 1 anomaly for periodic update\n\n# \u2500\u2500 Adaptation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nADAPT_INTERVAL_H  = 12      # hours between periodic training jobs\nBUFFER_CAPACITY   = 20_000  # max windows in D_low\nADAPT_BATCH       = 128\nADAPT_EPOCHS      = 10\n\n# \u2500\u2500 Inference / thresholding \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nTARGET_FPS        = 5\nANOMALY_TAU       = None    # None \u2192 calibrated from HPRS; float \u2192 fixed\nCONF_THRESH       = 0.4     # YOLO person confidence\n",
    "dataset/__init__.py": "",
    "dataset/pose_schema.py": "\"\"\"\n15-keypoint schema shared across the entire STG-NF pipeline.\n\nCOCO 17-pt raw indices\n  0:nose 1:L-eye 2:R-eye 3:L-ear 4:R-ear\n  5:L-shoulder 6:R-shoulder 7:L-elbow 8:R-elbow\n  9:L-wrist 10:R-wrist 11:L-hip 12:R-hip\n  13:L-knee 14:R-knee 15:L-ankle 16:R-ankle\n\nDerived 15-pt schema (facial landmarks removed, Neck / Chest derived)\n  0: Neck      = midpoint(L-shoulder, R-shoulder)\n  1: Chest     = midpoint(Neck, Mid-Hip)\n  2: LShoulder\n  3: RShoulder\n  4: LElbow\n  5: RElbow\n  6: LWrist\n  7: RWrist\n  8: LHip\n  9: RHip\n 10: MidHip    = midpoint(L-hip, R-hip)\n 11: LKnee\n 12: RKnee\n 13: LAnkle\n 14: RAnkle\n\nSpatial groups for alternating coupling masks\n  GROUP_A (upper body, 8 joints): 0-7\n  GROUP_B (lower body, 7 joints): 8-14\n\"\"\"\nfrom __future__ import annotations\nimport numpy as np\n\nNUM_KEYPOINTS = 15\n\nKEYPOINT_NAMES = [\n    \"Neck\", \"Chest\",\n    \"LShoulder\", \"RShoulder\",\n    \"LElbow\", \"RElbow\",\n    \"LWrist\", \"RWrist\",\n    \"LHip\", \"RHip\", \"MidHip\",\n    \"LKnee\", \"RKnee\",\n    \"LAnkle\", \"RAnkle\",\n]\n\n# Joint indices for normalisation\nNECK_IDX    = 0\nMID_HIP_IDX = 10\n\nGROUP_A = list(range(0, 8))    # upper body (8 joints)\nGROUP_B = list(range(8, 15))   # lower body (7 joints)\n\n# Skeleton edges (undirected) used for graph adjacency\nSKELETON_EDGES = [\n    (0, 1), (0, 2), (0, 3),   # Neck connections\n    (2, 4), (3, 5),            # shoulder\u2192elbow\n    (4, 6), (5, 7),            # elbow\u2192wrist\n    (1, 10),                   # Chest\u2192MidHip\n    (10, 8), (10, 9),          # MidHip\u2192hips\n    (8, 11), (9, 12),          # hip\u2192knee\n    (11, 13), (12, 14),        # knee\u2192ankle\n]\n\n# Limb pairs for QC (same set as edges)\nLIMB_PAIRS = SKELETON_EDGES\n\n\ndef build_15kp_from_coco(raw: np.ndarray) -> np.ndarray:\n    \"\"\"\n    Convert COCO-17 keypoints to our 15-kp schema.\n\n    Args:\n        raw: (17, 3)  [x, y, confidence]\n    Returns:\n        kps: (15, 3)\n    \"\"\"\n    assert raw.shape == (17, 3), raw.shape\n\n    def mid(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n        return np.array([(a[0]+b[0])/2, (a[1]+b[1])/2, (a[2]+b[2])/2],\n                        dtype=np.float32)\n\n    ls, rs  = raw[5],  raw[6]\n    lh, rh  = raw[11], raw[12]\n    neck    = mid(ls, rs)\n    mid_hip = mid(lh, rh)\n    chest   = mid(neck, mid_hip)\n\n    return np.stack([\n        neck, chest,\n        ls, rs,\n        raw[7], raw[8],   # elbows\n        raw[9], raw[10],  # wrists\n        lh, rh, mid_hip,\n        raw[13], raw[14], # knees\n        raw[15], raw[16], # ankles\n    ]).astype(np.float32)\n\n\ndef normalize_pose(kps: np.ndarray, torso_scale: float | None = None) -> np.ndarray:\n    \"\"\"\n    Center on MidHip; scale by median torso length (Neck\u2013MidHip distance).\n\n    Args:\n        kps:         (T, 15, 2) \u2013 x/y only, already stripped of confidence\n        torso_scale: if provided, use this fixed scale (for test-time consistency)\n    Returns:\n        norm_kps:    (T, 15, 2)\n        scale:       float \u2013 torso scale used (useful to cache at test time)\n    \"\"\"\n    origin = kps[:, MID_HIP_IDX:MID_HIP_IDX+1, :]   # (T, 1, 2)\n    centred = kps - origin\n\n    if torso_scale is None:\n        neck    = centred[:, NECK_IDX, :]             # (T, 2)\n        lengths = np.linalg.norm(neck, axis=1)        # (T,)\n        torso_scale = float(np.median(lengths)) or 1.0\n\n    return (centred / torso_scale).astype(np.float32), torso_scale\n",
    "dataset/retail_dataset.py": "\"\"\"\nRetailS dataset loading and windowing.\n\nActual on-disk layout\n---------------------\nRetailS/\n  RetailS_train/\n    pose/train/\n      <clip_stem>.json        \u2190 normal training clips, no labels\n  RetailS_test_staged/\n    pose/test/<clip_stem>.json\n    gt/test_frame_mask/<clip_stem>.npy\n  RetailS_test_realworld/\n    pose/test/<clip_stem>.json\n    gt/test_frame_mask/<clip_stem>.npy\n\nJSON schema\n-----------\n{\n  \"<person_id>\": {                  \u2190 string integer, e.g. \"1\"\n    \"<frame_idx>\": {                \u2190 string integer, e.g. \"0\"\n      \"keypoints\": [x0,y0,c0, ...] \u2190 flat list of 17\u00d73 = 51 floats (COCO-17)\n    }\n  }\n}\n\nLabel .npy\n----------\n  1-D uint8 array of length N_frames  (0=normal, 1=shoplifting)\n  Indexable by frame_idx (0-based).\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import List, Optional, Tuple\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom config import WINDOW_T\nfrom dataset.pose_schema import NUM_KEYPOINTS, build_15kp_from_coco, normalize_pose\n\n\n# \u2500\u2500 helpers \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef _load_json(path: Path) -> dict:\n    with open(path) as f:\n        return json.load(f)\n\n\ndef _parse_kps_flat(flat: list) -> np.ndarray:\n    \"\"\"[x0,y0,c0, x1,y1,c1, ...] \u2192 (17, 3) float32.\"\"\"\n    return np.array(flat, dtype=np.float32).reshape(17, 3)\n\n\ndef _person_sequence(clip: dict, pid: str) -> np.ndarray:\n    \"\"\"\n    Extract one person's frames as (T, 15, 3).\n\n    clip: parsed JSON dict  {person_id_str: {frame_idx_str: {keypoints:[...]}}}\n    pid:  string person id\n    \"\"\"\n    person = clip.get(pid, {})\n    if not person:\n        return np.empty((0, NUM_KEYPOINTS, 3), dtype=np.float32)\n\n    frames = sorted(person.items(), key=lambda kv: int(kv[0]))\n    kps = []\n    for _, fdata in frames:\n        kp17 = _parse_kps_flat(fdata[\"keypoints\"])   # (17, 3)\n        kp15 = build_15kp_from_coco(kp17)            # (15, 3)\n        kps.append(kp15)\n\n    return np.stack(kps, axis=0)   # (T, 15, 3)\n\n\ndef _frame_indices(clip: dict, pid: str) -> List[int]:\n    \"\"\"Sorted list of integer frame indices for a person.\"\"\"\n    return sorted(int(k) for k in clip.get(pid, {}).keys())\n\n\ndef _windows(\n    seq: np.ndarray,          # (T, 15, 3)\n    labels: Optional[np.ndarray],  # (L,) uint8 or None\n    fidxs: List[int],          # frame indices matching seq rows\n    window: int,\n    stride: int,\n) -> List[Tuple[np.ndarray, Optional[int]]]:\n    \"\"\"\n    Slide a window over seq; return (window_kps (W,15,3), window_label).\n    window_label = max label over the window frames (any shoplifting \u2192 anomalous).\n    \"\"\"\n    T = seq.shape[0]\n    if T < window:\n        return []\n\n    result = []\n    for start in range(0, T - window + 1, stride):\n        end   = start + window\n        w     = seq[start:end]    # (W, 15, 3)\n        lab   = None\n        if labels is not None:\n            frame_slice = [fidxs[i] for i in range(start, end)\n                           if fidxs[i] < len(labels)]\n            if frame_slice:\n                lab = int(labels[frame_slice].max())\n            else:\n                lab = 0\n        result.append((w, lab))\n    return result\n\n\n# \u2500\u2500 datasets \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass RetailSTrainDataset(Dataset):\n    \"\"\"\n    Normal-only training set from RetailS_train/pose/train/*.json.\n    Each sample: (2, T, V) = (2, 24, 15) normalised pose tensor.\n    \"\"\"\n\n    def __init__(\n        self,\n        retail_root: Path,\n        window: int = WINDOW_T,\n        stride: int = 6,\n        max_clips: Optional[int] = None,\n    ) -> None:\n        self.window = window\n        self.stride = stride\n        self._samples: List[np.ndarray] = []   # each (T, 15, 2)\n\n        train_dir = Path(retail_root) / \"RetailS_train\" / \"pose\" / \"train\"\n        if not train_dir.exists():\n            raise FileNotFoundError(f\"Train dir not found: {train_dir}\")\n\n        clips = sorted(train_dir.glob(\"*.json\"))\n        if max_clips:\n            clips = clips[:max_clips]\n\n        print(f\"[RetailSTrainDataset] Loading {len(clips)} clip files \u2026\")\n        for p in clips:\n            self._ingest(p)\n\n        print(f\"[RetailSTrainDataset] {len(self._samples):,} normal windows\")\n\n    def _ingest(self, path: Path) -> None:\n        clip = _load_json(path)\n        for pid in clip.keys():\n            seq   = _person_sequence(clip, pid)       # (T, 15, 3)\n            fidxs = _frame_indices(clip, pid)\n            for (w, _) in _windows(seq, None, fidxs, self.window, self.stride):\n                xy = w[:, :, :2]                      # (T, 15, 2)\n                norm_xy, _ = normalize_pose(xy)\n                self._samples.append(norm_xy)\n\n    def __len__(self) -> int:\n        return len(self._samples)\n\n    def __getitem__(self, idx: int) -> torch.Tensor:\n        return torch.from_numpy(self._samples[idx]).permute(2, 0, 1)  # (2, T, V)\n\n\nclass RetailSTestDataset(Dataset):\n    \"\"\"\n    Labelled test set for threshold calibration and evaluation,\n    and as the anomaly source for 9:1 mixing.\n\n    split:        'staged' | 'real'\n    anomaly_only: if True, return only windows with label=1 (shoplifting)\n    \"\"\"\n\n    def __init__(\n        self,\n        retail_root: Path,\n        split: str = \"staged\",\n        window: int = WINDOW_T,\n        stride: int = 6,\n        anomaly_only: bool = False,\n    ) -> None:\n        assert split in {\"staged\", \"real\"}, f\"Unknown split: {split}\"\n        folder_map = {\"staged\": \"RetailS_test_staged\", \"real\": \"RetailS_test_realworld\"}\n        folder     = folder_map[split]\n\n        pose_dir  = Path(retail_root) / folder / \"pose\" / \"test\"\n        label_dir = Path(retail_root) / folder / \"gt\" / \"test_frame_mask\"\n\n        self._samples: List[np.ndarray] = []\n        self._labels:  List[int]        = []\n\n        for json_path in sorted(pose_dir.glob(\"*.json\")):\n            npy_path = label_dir / f\"{json_path.stem}.npy\"\n            if not npy_path.exists():\n                continue\n            self._ingest(json_path, npy_path, anomaly_only)\n\n        tag = f\"{split}/anomaly_only\" if anomaly_only else split\n        print(f\"[RetailSTestDataset/{tag}] {len(self._samples):,} windows \"\n              f\"({sum(self._labels)} anomalous)\")\n\n    def _ingest(self, json_path: Path, npy_path: Path,\n                anomaly_only: bool) -> None:\n        clip   = _load_json(json_path)\n        labels = np.load(npy_path)   # (N_frames,) uint8\n\n        for pid in clip.keys():\n            seq   = _person_sequence(clip, pid)\n            fidxs = _frame_indices(clip, pid)\n\n            for (w, lab) in _windows(seq, labels, fidxs, WINDOW_T, 6):\n                if lab is None:\n                    lab = 0\n                if anomaly_only and lab == 0:\n                    continue\n                xy = w[:, :, :2]\n                norm_xy, _ = normalize_pose(xy)\n                self._samples.append(norm_xy)\n                self._labels.append(lab)\n\n    def __len__(self) -> int:\n        return len(self._samples)\n\n    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:\n        x = torch.from_numpy(self._samples[idx]).permute(2, 0, 1)  # (2, T, V)\n        return x, self._labels[idx]\n\n\nclass WebcamWindowDataset(Dataset):\n    \"\"\"\n    In-memory wrapper for D_low windows used by the adaptation pipeline.\n    windows: list of (T, 15, 2) float32 arrays.\n    \"\"\"\n\n    def __init__(self, windows: List[np.ndarray]) -> None:\n        self._data = windows\n\n    def __len__(self) -> int:\n        return len(self._data)\n\n    def __getitem__(self, idx: int) -> torch.Tensor:\n        return torch.from_numpy(self._data[idx]).permute(2, 0, 1)  # (2, T, V)\n",
    "model/__init__.py": "",
    "model/graph.py": "\"\"\"\nSkeleton graph adjacency matrices for the 15-keypoint schema.\n\nTwo adjacency strategies are provided:\n  - binary:     A[i,j] = 1 if (i,j) is a skeleton edge (self-loops included)\n  - normalised: D^{-1/2} A D^{-1/2}  (symmetric normalisation)\n\nBoth return NumPy arrays of shape (V, V).\n\"\"\"\nfrom __future__ import annotations\nimport numpy as np\nimport torch\n\nfrom dataset.pose_schema import NUM_KEYPOINTS, SKELETON_EDGES\n\n\ndef build_adjacency(\n    strategy: str = \"normalised\",\n    add_self_loops: bool = True,\n) -> np.ndarray:\n    \"\"\"\n    Build (V, V) adjacency matrix.\n\n    Args:\n        strategy:        'binary' | 'normalised'\n        add_self_loops:  add identity to A before normalisation\n    Returns:\n        A: (V, V) float32\n    \"\"\"\n    V = NUM_KEYPOINTS\n    A = np.zeros((V, V), dtype=np.float32)\n\n    for (i, j) in SKELETON_EDGES:\n        A[i, j] = 1.0\n        A[j, i] = 1.0  # undirected\n\n    if add_self_loops:\n        A += np.eye(V, dtype=np.float32)\n\n    if strategy == \"binary\":\n        return A\n\n    if strategy == \"normalised\":\n        # D^{-1/2} A D^{-1/2}\n        d   = A.sum(axis=1)\n        d_inv_sqrt = np.where(d > 0, 1.0 / np.sqrt(d), 0.0)\n        D_inv_sqrt = np.diag(d_inv_sqrt)\n        return (D_inv_sqrt @ A @ D_inv_sqrt).astype(np.float32)\n\n    raise ValueError(f\"Unknown adjacency strategy: {strategy}\")\n\n\ndef adjacency_tensor(strategy: str = \"normalised\") -> torch.Tensor:\n    \"\"\"Return the adjacency matrix as a float32 torch.Tensor (V, V).\"\"\"\n    return torch.from_numpy(build_adjacency(strategy))\n",
    "model/stgcn.py": "\"\"\"\nSpatio-Temporal Graph Convolution Network building blocks.\n\nThese act as the *coupling network* inside each affine coupling layer of the\nSTG-NF flow: they take a subset of joints as input and produce per-joint\nscale (s) and shift (t) tensors for the complementary joint subset.\n\nShapes throughout:  (B, C, T, V)\n  B \u2013 batch\n  C \u2013 channels (starts at COORDS=2, projected to HIDDEN_CHANNELS)\n  T \u2013 time steps (WINDOW_T = 24)\n  V \u2013 joints (NUM_KEYPOINTS = 15, or a subset)\n\"\"\"\nfrom __future__ import annotations\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom model.graph import adjacency_tensor\nfrom dataset.pose_schema import NUM_KEYPOINTS\n\n\nclass SpatialGraphConv(nn.Module):\n    \"\"\"\n    Single-partition spatial graph convolution:\n        Y = \u03c3( W  * (A_hat \u2297 X) )\n\n    A_hat is registered as a non-trainable buffer so it moves with the module\n    to whatever device the model lives on.\n    \"\"\"\n\n    def __init__(self, in_channels: int, out_channels: int, V: int = NUM_KEYPOINTS) -> None:\n        super().__init__()\n        A = adjacency_tensor(\"normalised\")          # (V, V) \u2013 full skeleton\n        # Subgraph: take the top-left V\u00d7V block if V < NUM_KEYPOINTS\n        self.register_buffer(\"A\", A[:V, :V])\n        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        # x: (B, C, T, V)\n        # A: (V, V)\n        # out: (B, C_out, T, V)\n        support = torch.einsum(\"bctn,nm->bctm\", x, self.A)\n        return self.conv(support)\n\n\nclass TemporalConv(nn.Module):\n    \"\"\"\n    Temporal (1-D) convolution over the time axis with optional dilation.\n    Kernel is applied per-joint independently (grouped conv = depthwise along V).\n    \"\"\"\n\n    def __init__(\n        self,\n        channels: int,\n        kernel_size: int = 3,\n        dilation: int = 1,\n    ) -> None:\n        super().__init__()\n        pad = (kernel_size - 1) * dilation // 2\n        self.conv = nn.Conv2d(\n            channels, channels,\n            kernel_size=(kernel_size, 1),\n            padding=(pad, 0),\n            dilation=(dilation, 1),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.conv(x)\n\n\nclass STGCNBlock(nn.Module):\n    \"\"\"\n    One ST-GCN block: SpatialGraphConv \u2192 BN \u2192 ReLU \u2192 TemporalConv \u2192 BN \u2192 ReLU\n    with an optional residual projection.\n    \"\"\"\n\n    def __init__(\n        self,\n        in_channels: int,\n        out_channels: int,\n        V: int = NUM_KEYPOINTS,\n        temporal_kernel: int = 3,\n        dilation: int = 1,\n        dropout: float = 0.0,\n    ) -> None:\n        super().__init__()\n        self.spatial = SpatialGraphConv(in_channels, out_channels, V)\n        self.bn_s    = nn.BatchNorm2d(out_channels)\n        self.temporal = TemporalConv(out_channels, temporal_kernel, dilation)\n        self.bn_t    = nn.BatchNorm2d(out_channels)\n        self.drop    = nn.Dropout(dropout) if dropout > 0 else nn.Identity()\n\n        self.residual = (\n            nn.Conv2d(in_channels, out_channels, kernel_size=1)\n            if in_channels != out_channels\n            else nn.Identity()\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        res = self.residual(x)\n        out = F.relu(self.bn_s(self.spatial(x)))\n        out = F.relu(self.bn_t(self.temporal(out)))\n        return self.drop(out + res)\n\n\nclass CouplingNetwork(nn.Module):\n    \"\"\"\n    Coupling network used inside each affine coupling layer.\n\n    Takes a (B, C_in, T, V_in) input (the 'conditioner' joint group) and\n    produces (s, t) each of shape (B, C_out, T, V_out) for the 'target' group.\n\n    Two ST-GCN blocks followed by a 1\u00d71 conv to output 2\u00d7C_out channels\n    (first half = log-scale s, second half = shift t).\n\n    The tanh on s keeps log-scale bounded, preventing exploding determinants.\n    \"\"\"\n\n    def __init__(\n        self,\n        in_channels: int,\n        out_channels: int,\n        hidden: int,\n        V_in: int,\n        V_out: int,\n    ) -> None:\n        super().__init__()\n        self.V_out = V_out\n        self.C_out = out_channels\n\n        self.net = nn.Sequential(\n            STGCNBlock(in_channels, hidden, V=V_in),\n            STGCNBlock(hidden,      hidden, V=V_in),\n        )\n        # Project from V_in joints to V_out joints via MLP on last dim\n        self.joint_proj = nn.Linear(V_in, V_out)\n        # Output: 2 * out_channels (s and t interleaved)\n        self.out_conv = nn.Conv2d(hidden, 2 * out_channels, kernel_size=1)\n\n        # Initialise output to zero so the flow starts as identity\n        nn.init.zeros_(self.out_conv.weight)\n        nn.init.zeros_(self.out_conv.bias)\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        \"\"\"\n        Args:\n            x: (B, C_in, T, V_in)\n        Returns:\n            s: (B, C_out, T, V_out)  log-scale\n            t: (B, C_out, T, V_out)  shift\n        \"\"\"\n        h = self.net(x)                             # (B, hidden, T, V_in)\n        h = self.joint_proj(h)                      # (B, hidden, T, V_out)\n        st = self.out_conv(h)                       # (B, 2*C_out, T, V_out)\n        s_raw, t = st.chunk(2, dim=1)               # each (B, C_out, T, V_out)\n        s = torch.tanh(s_raw)                       # bounded log-scale\n        return s, t\n",
    "model/flows.py": "\"\"\"\nNormalizing Flow building blocks for the STG-NF model.\n\nThree primitives:\n  ActNorm          \u2013 data-dependent scale/shift initialisation (per-channel)\n  InvertibleLinear \u2013 LU-decomposed 1\u00d71 invertible convolution (spatial mixing)\n  SpatialCoupling  \u2013 affine coupling with spatial body-part split + ST-GCN net\n\nAll modules expose:\n  forward(x)  \u2192 (z, log_det)    [density estimation direction]\n  inverse(z)  \u2192 x               [generation direction, unused at inference]\n\"\"\"\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom dataset.pose_schema import GROUP_A, GROUP_B\nfrom model.stgcn import CouplingNetwork\n\n\n# \u2500\u2500 helpers \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef _sum_except_batch(t: torch.Tensor) -> torch.Tensor:\n    \"\"\"Sum over all dimensions except batch (dim 0).\"\"\"\n    return t.reshape(t.shape[0], -1).sum(dim=1)\n\n\n# \u2500\u2500 ActNorm \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass ActNorm(nn.Module):\n    \"\"\"\n    Activation normalisation (Glow \u00a73.1).\n\n    Learnable per-channel scale and bias initialised so that the first batch\n    has zero mean and unit variance (data-dependent init).\n\n    Input/output: (B, C, T, V)\n    log_det contribution: T*V * sum(log |scale|)\n    \"\"\"\n\n    def __init__(self, num_channels: int) -> None:\n        super().__init__()\n        self.num_channels = num_channels\n        self.register_parameter(\"log_scale\", nn.Parameter(torch.zeros(1, num_channels, 1, 1)))\n        self.register_parameter(\"bias\",      nn.Parameter(torch.zeros(1, num_channels, 1, 1)))\n        self._initialised = False\n\n    @torch.no_grad()\n    def _data_init(self, x: torch.Tensor) -> None:\n        # x: (B, C, T, V)\n        mean = x.mean(dim=(0, 2, 3), keepdim=True)          # (1, C, 1, 1)\n        std  = x.std(dim=(0, 2, 3), keepdim=True).clamp(min=1e-6)\n        self.bias.data      = -mean\n        self.log_scale.data = -std.log()\n        self._initialised   = True\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        if not self._initialised:\n            self._data_init(x)\n\n        z = (x + self.bias) * self.log_scale.exp()\n\n        # log|det J| = T*V * sum(log|scale|)\n        _, _, T, V = x.shape\n        log_det = _sum_except_batch(self.log_scale).expand(x.shape[0]) * T * V\n        return z, log_det\n\n    def inverse(self, z: torch.Tensor) -> torch.Tensor:\n        return z * (-self.log_scale).exp() - self.bias\n\n\n# \u2500\u2500 InvertibleLinear \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass InvertibleLinear(nn.Module):\n    \"\"\"\n    Learnable 1\u00d71 invertible convolution via LU decomposition (Glow \u00a73.2).\n\n    Mixes channels at every spatial/temporal position.\n    Input/output: (B, C, T, V)\n    log_det: T*V * log|det(W)|\n    \"\"\"\n\n    def __init__(self, num_channels: int) -> None:\n        super().__init__()\n        # Initialise with a random orthogonal matrix (stable determinant = \u00b11)\n        W_init, _ = torch.linalg.qr(torch.randn(num_channels, num_channels))\n        self.register_parameter(\"W_raw\", nn.Parameter(W_init))\n\n    @property\n    def W(self) -> torch.Tensor:\n        return self.W_raw\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        # x: (B, C, T, V)\n        W = self.W                                        # (C, C)\n        z = F.conv2d(x, W.unsqueeze(-1).unsqueeze(-1))   # 1\u00d71 conv\n\n        _, _, T, V = x.shape\n        # slogdet must run on CPU (MPS has no float64 support)\n        log_det = (torch.slogdet(W.cpu().float())[1]\n                   .to(x.device)\n                   .expand(x.shape[0]) * T * V)\n        return z, log_det\n\n    def inverse(self, z: torch.Tensor) -> torch.Tensor:\n        W_inv = torch.linalg.inv(self.W)\n        return F.conv2d(z, W_inv.unsqueeze(-1).unsqueeze(-1))\n\n\n# \u2500\u2500 SpatialCoupling \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass SpatialCoupling(nn.Module):\n    \"\"\"\n    Affine coupling layer that splits along the *spatial joint* dimension.\n\n    Alternating layers use GROUP_A\u2192B and GROUP_B\u2192A partitions:\n      - Even layers: condition on GROUP_A joints, transform GROUP_B joints\n      - Odd  layers: condition on GROUP_B joints, transform GROUP_A joints\n\n    For a window of shape (B, C, T, V):\n      x_a = x[..., group_cond]   # conditioning joints\n      x_b = x[..., group_tgt]    # target joints\n      s, t = coupling_net(x_a)\n      z_b  = x_b * exp(s) + t\n      log_det = sum(s)\n\n    The full output z is reconstructed by re-inserting z_b at group_tgt indices.\n    \"\"\"\n\n    def __init__(\n        self,\n        channels: int,\n        hidden: int,\n        layer_idx: int,\n    ) -> None:\n        super().__init__()\n        # Alternating groups\n        if layer_idx % 2 == 0:\n            self.cond_idx = torch.tensor(GROUP_A, dtype=torch.long)  # 8 joints\n            self.tgt_idx  = torch.tensor(GROUP_B, dtype=torch.long)  # 7 joints\n        else:\n            self.cond_idx = torch.tensor(GROUP_B, dtype=torch.long)\n            self.tgt_idx  = torch.tensor(GROUP_A, dtype=torch.long)\n\n        self.coupling = CouplingNetwork(\n            in_channels  = channels,\n            out_channels = channels,\n            hidden       = hidden,\n            V_in         = len(self.cond_idx),\n            V_out        = len(self.tgt_idx),\n        )\n\n    def _move_indices(self, device: torch.device) -> None:\n        \"\"\"Move index tensors to the right device (lazy, once).\"\"\"\n        if self.cond_idx.device != device:\n            self.cond_idx = self.cond_idx.to(device)\n            self.tgt_idx  = self.tgt_idx.to(device)\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        self._move_indices(x.device)\n        x_a = x.index_select(-1, self.cond_idx)    # (B, C, T, V_cond)\n        x_b = x.index_select(-1, self.tgt_idx)     # (B, C, T, V_tgt)\n\n        s, t = self.coupling(x_a)                   # each (B, C, T, V_tgt)\n        z_b  = x_b * s.exp() + t\n\n        log_det = _sum_except_batch(s)\n\n        # Reconstruct full tensor preserving joint order\n        z = x.clone()\n        z.index_copy_(-1, self.tgt_idx, z_b)\n        return z, log_det\n\n    def inverse(self, z: torch.Tensor) -> torch.Tensor:\n        self._move_indices(z.device)\n        x_a = z.index_select(-1, self.cond_idx)\n        z_b = z.index_select(-1, self.tgt_idx)\n\n        s, t = self.coupling(x_a)\n        x_b  = (z_b - t) * (-s).exp()\n\n        x = z.clone()\n        x.index_copy_(-1, self.tgt_idx, x_b)\n        return x\n",
    "model/stg_nf.py": "\"\"\"\nSTG-NF: Spatio-Temporal Graph Normalizing Flow for unsupervised anomaly detection.\n\nThe model learns a bijective mapping from normal pose trajectories to a standard\nGaussian distribution by training on normal data only.\n\nArchitecture (one \"step\" repeated FLOW_DEPTH times):\n  ActNorm  \u2192  InvertibleLinear  \u2192  SpatialCoupling\n\nTraining objective:\n  min  -log p(z)  -  log|det J|\n  = min  0.5 * ||z||^2  -  log_det_sum\n         ^^^^^^^^^^^       ^^^^^^^^^^^\n         Gaussian NLL       flow log-det\n\nAnomaly score at inference:\n  score(x) = -log p(x)   (higher = more anomalous)\n  normality_score = exp(-score / D)  \u2208 (0, 1]  (higher = more normal)\n  where D = total input dimensionality (C * T * V)\n\"\"\"\nfrom __future__ import annotations\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport numpy as np\n\nfrom config import FLOW_DEPTH, HIDDEN_CHANNELS, COORDS, WINDOW_T, NUM_JOINTS\nfrom model.flows import ActNorm, InvertibleLinear, SpatialCoupling\n\nLOG_2PI = float(np.log(2 * np.pi))\nINPUT_DIM = COORDS * WINDOW_T * NUM_JOINTS  # 2 * 24 * 15 = 720\n\n\nclass FlowStep(nn.Module):\n    \"\"\"ActNorm \u2192 InvertibleLinear \u2192 SpatialCoupling (one step of the flow).\"\"\"\n\n    def __init__(self, channels: int, hidden: int, step_idx: int) -> None:\n        super().__init__()\n        self.actnorm = ActNorm(channels)\n        self.inv_lin = InvertibleLinear(channels)\n        self.coupling = SpatialCoupling(channels, hidden, layer_idx=step_idx)\n\n    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        z, ld1 = self.actnorm(x)\n        z, ld2 = self.inv_lin(z)\n        z, ld3 = self.coupling(z)\n        return z, ld1 + ld2 + ld3\n\n    def inverse(self, z: torch.Tensor) -> torch.Tensor:\n        x = self.coupling.inverse(z)\n        x = self.inv_lin.inverse(x)\n        x = self.actnorm.inverse(x)\n        return x\n\n\nclass STGNF(nn.Module):\n    \"\"\"\n    Spatio-Temporal Graph Normalizing Flow model.\n\n    Input/output shape:  (B, C, T, V) = (B, 2, 24, 15)\n    \"\"\"\n\n    def __init__(\n        self,\n        flow_depth: int   = FLOW_DEPTH,\n        hidden:     int   = HIDDEN_CHANNELS,\n        channels:   int   = COORDS,\n    ) -> None:\n        super().__init__()\n        self.input_dim = INPUT_DIM\n\n        self.steps = nn.ModuleList([\n            FlowStep(channels, hidden, i) for i in range(flow_depth)\n        ])\n\n    # \u2500\u2500 density estimation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\n    def forward(\n        self, x: torch.Tensor\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        \"\"\"\n        Args:\n            x: (B, C, T, V)  normalised pose trajectory\n\n        Returns:\n            z:       (B, C, T, V)  Gaussian-distributed latent\n            log_det: (B,)         sum of log|det J| across all steps\n        \"\"\"\n        log_det = torch.zeros(x.shape[0], device=x.device)\n        z = x\n        for step in self.steps:\n            z, ld = step(z)\n            log_det = log_det + ld\n        return z, log_det\n\n    def log_prob(self, x: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Returns log p(x) per sample.  Shape: (B,)\n\n        log p(x) = log p_z(z) + log|det J|\n                 = -0.5*(||z||^2 + D*log(2\u03c0)) + log_det\n        \"\"\"\n        z, log_det = self.forward(x)\n        z_flat   = z.reshape(z.shape[0], -1)              # (B, D)\n        log_pz   = -0.5 * (z_flat.pow(2).sum(dim=1) + self.input_dim * LOG_2PI)\n        return log_pz + log_det\n\n    def nll_loss(self, x: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Mean negative log-likelihood over the batch.\n        Normalised by input dimensionality for stable magnitude across configs.\n        \"\"\"\n        return -self.log_prob(x).mean() / self.input_dim\n\n    # \u2500\u2500 scoring \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\n    @torch.no_grad()\n    def anomaly_score(self, x: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Unnormalised anomaly score: -log p(x).  Shape: (B,)\n        Higher value \u2192 more anomalous.\n        \"\"\"\n        return -self.log_prob(x)\n\n    @torch.no_grad()\n    def normality_score(self, x: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Normality score \u2208 (0, 1].  Shape: (B,)\n        Higher \u2192 more normal.  Suitable for real-time overlay.\n        \"\"\"\n        neg_ll = self.anomaly_score(x) / self.input_dim\n        return torch.exp(-neg_ll.clamp(min=0))\n\n    # \u2500\u2500 generation (inverse, for debugging) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\n    @torch.no_grad()\n    def sample(self, n: int, device: torch.device) -> torch.Tensor:\n        \"\"\"Generate `n` random pose trajectories from the learnt distribution.\"\"\"\n        z = torch.randn(n, COORDS, WINDOW_T, NUM_JOINTS, device=device)\n        x = z\n        for step in reversed(self.steps):\n            x = step.inverse(x)\n        return x\n\n\ndef build_model(\n    flow_depth: int = FLOW_DEPTH,\n    hidden:     int = HIDDEN_CHANNELS,\n    device: Optional[torch.device] = None,\n) -> STGNF:\n    \"\"\"Convenience factory. Places model on `device`.\"\"\"\n    model = STGNF(flow_depth=flow_depth, hidden=hidden)\n    if device is not None:\n        model = model.to(device)\n    return model\n",
    "training/__init__.py": "",
    "training/trainer.py": "\"\"\"\nTraining and fine-tuning loop for the STG-NF model.\n\nResponsibilities:\n  - Standard training epoch on normal data (NLL minimisation)\n  - Periodic adaptation training using 9:1 mixed data\n  - Checkpoint save / load\n  - MPS / CUDA / CPU device routing\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom pathlib import Path\nfrom typing import Optional\n\nimport torch\nimport torch.optim as optim\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm import tqdm\n\nfrom config import (\n    BATCH_SIZE, CHECKPOINT_DIR, GRAD_CLIP, LR, MAX_EPOCHS,\n    WEIGHT_DECAY, ADAPT_BATCH, ADAPT_EPOCHS,\n)\nfrom model.stg_nf import STGNF\n\n\ndef _select_device() -> torch.device:\n    if torch.backends.mps.is_available():\n        return torch.device(\"mps\")\n    if torch.cuda.is_available():\n        return torch.device(\"cuda\")\n    return torch.device(\"cpu\")\n\n\n# \u2500\u2500 helpers \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef save_checkpoint(\n    model: STGNF,\n    optimizer: optim.Optimizer,\n    epoch: int,\n    best_loss: float,\n    path: Path,\n) -> None:\n    torch.save(\n        {\n            \"epoch\":     epoch,\n            \"best_loss\": best_loss,\n            \"model\":     model.state_dict(),\n            \"optimizer\": optimizer.state_dict(),\n        },\n        path,\n    )\n\n\ndef load_checkpoint(\n    model: STGNF,\n    path: Path,\n    optimizer: Optional[optim.Optimizer] = None,\n    device: Optional[torch.device] = None,\n) -> tuple[int, float]:\n    \"\"\"\n    Load weights (and optionally optimizer state) from a checkpoint.\n    Returns (epoch, best_loss).\n    \"\"\"\n    ckpt = torch.load(path, map_location=device or \"cpu\")\n    model.load_state_dict(ckpt[\"model\"])\n    if optimizer and \"optimizer\" in ckpt:\n        optimizer.load_state_dict(ckpt[\"optimizer\"])\n    return ckpt.get(\"epoch\", 0), ckpt.get(\"best_loss\", float(\"inf\"))\n\n\n# \u2500\u2500 training loop \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass Trainer:\n    \"\"\"\n    Handles one full training run: initial training on normal-only data,\n    or a shorter adaptation round on mixed data.\n    \"\"\"\n\n    def __init__(\n        self,\n        model:       STGNF,\n        device:      Optional[torch.device] = None,\n        lr:          float = LR,\n        weight_decay: float = WEIGHT_DECAY,\n        max_epochs:  int   = MAX_EPOCHS,\n        ckpt_dir:    Path  = CHECKPOINT_DIR,\n        run_name:    str   = \"stgnf\",\n    ) -> None:\n        self.model       = model\n        self.device      = device or _select_device()\n        self.max_epochs  = max_epochs\n        self.ckpt_dir    = Path(ckpt_dir)\n        self.run_name    = run_name\n\n        self.model.to(self.device)\n\n        self.optimizer = optim.AdamW(\n            model.parameters(), lr=lr, weight_decay=weight_decay\n        )\n        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(\n            self.optimizer, T_max=max_epochs, eta_min=lr * 0.01\n        )\n\n        self.best_loss   = float(\"inf\")\n        self.start_epoch = 0\n        self._metrics: list[dict] = []\n\n    # \u2500\u2500 public API \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\n    def fit(\n        self,\n        train_loader: DataLoader,\n        val_loader:   Optional[DataLoader] = None,\n        resume_from:  Optional[Path]       = None,\n    ) -> None:\n        \"\"\"Full training run.\"\"\"\n        if resume_from and resume_from.exists():\n            self.start_epoch, self.best_loss = load_checkpoint(\n                self.model, resume_from, self.optimizer, self.device\n            )\n            print(f\"[trainer] Resumed from {resume_from} (epoch {self.start_epoch})\")\n\n        for epoch in range(self.start_epoch, self.max_epochs):\n            t0       = time.time()\n            train_loss = self._epoch(train_loader, training=True)\n            val_loss   = self._epoch(val_loader,   training=False) if val_loader else None\n            self.scheduler.step()\n\n            elapsed = time.time() - t0\n            record  = {\n                \"epoch\": epoch,\n                \"train_nll\": train_loss,\n                \"val_nll\":   val_loss,\n                \"lr\":        self.scheduler.get_last_lr()[0],\n                \"elapsed_s\": elapsed,\n            }\n            self._metrics.append(record)\n            self._log(record)\n\n            # Save best checkpoint\n            metric = val_loss if val_loss is not None else train_loss\n            if metric < self.best_loss:\n                self.best_loss = metric\n                self._save(\"best.pt\", epoch)\n\n            # Periodic checkpoint every 10 epochs\n            if (epoch + 1) % 10 == 0:\n                self._save(f\"epoch_{epoch+1:04d}.pt\", epoch)\n\n        self._save(\"last.pt\", self.max_epochs - 1)\n        self._dump_metrics()\n\n    def adapt(\n        self,\n        mixed_loader: DataLoader,\n        epochs: int = ADAPT_EPOCHS,\n        out_path: Optional[Path] = None,\n    ) -> Path:\n        \"\"\"\n        Short fine-tuning pass for the Periodic Adaptation Pipeline.\n        Saves weights to `out_path` (default: ckpt_dir/adapted_candidate.pt).\n        Returns the path so the caller can perform an atomic swap.\n        \"\"\"\n        if out_path is None:\n            out_path = self.ckpt_dir / \"adapted_candidate.pt\"\n\n        for epoch in range(epochs):\n            loss = self._epoch(mixed_loader, training=True)\n            print(f\"[adapt] epoch {epoch+1}/{epochs}  NLL={loss:.6f}\")\n\n        torch.save(self.model.state_dict(), out_path)\n        print(f\"[adapt] Candidate weights saved to {out_path}\")\n        return out_path\n\n    # \u2500\u2500 internals \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\n    def _epoch(\n        self,\n        loader: Optional[DataLoader],\n        training: bool,\n    ) -> float:\n        if loader is None:\n            return float(\"nan\")\n\n        self.model.train(training)\n        total_loss = 0.0\n        n_batches  = 0\n\n        ctx = torch.enable_grad() if training else torch.no_grad()\n        with ctx:\n            pbar = tqdm(loader, desc=\"train\" if training else \"val \", leave=False)\n            for batch in pbar:\n                # Accept plain tensors or (tensor, *) tuples from labelled datasets\n                x = batch[0] if isinstance(batch, (list, tuple)) else batch\n                x = x.to(self.device, non_blocking=True)\n                loss = self.model.nll_loss(x)\n\n                if training:\n                    self.optimizer.zero_grad(set_to_none=True)\n                    loss.backward()\n                    torch.nn.utils.clip_grad_norm_(\n                        self.model.parameters(), GRAD_CLIP\n                    )\n                    self.optimizer.step()\n\n                total_loss += float(loss.detach())\n                n_batches  += 1\n                pbar.set_postfix(nll=f\"{float(loss.detach()):.4f}\")\n\n        return total_loss / max(n_batches, 1)\n\n    def _save(self, name: str, epoch: int) -> None:\n        path = self.ckpt_dir / f\"{self.run_name}_{name}\"\n        save_checkpoint(self.model, self.optimizer, epoch, self.best_loss, path)\n\n    @staticmethod\n    def _log(record: dict) -> None:\n        parts = [f\"epoch={record['epoch']}\",\n                 f\"train={record['train_nll']:.6f}\"]\n        if record[\"val_nll\"] is not None:\n            parts.append(f\"val={record['val_nll']:.6f}\")\n        parts.append(f\"lr={record['lr']:.2e}\")\n        parts.append(f\"t={record['elapsed_s']:.1f}s\")\n        print(\"[trainer] \" + \"  \".join(parts))\n\n    def _dump_metrics(self) -> None:\n        out = self.ckpt_dir / f\"{self.run_name}_metrics.json\"\n        with open(out, \"w\") as f:\n            json.dump(self._metrics, f, indent=2)\n        print(f\"[trainer] Metrics saved to {out}\")\n",
    "training/mixing.py": "\"\"\"\n9:1 normal-to-anomaly data mixing for periodic adaptation training.\n\nThe RetailS staged subset provides labelled shoplifting windows.\nFor each adaptation round, a DataLoader is built that yields mini-batches\nwith exactly 9 normal samples for every 1 anomaly sample.\n\nThis class is *also* used during initial training when --mix flag is set,\nto provide the model a weak signal about what NOT to model normally.\n\"\"\"\nfrom __future__ import annotations\nfrom typing import List\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Dataset, WeightedRandomSampler\n\n\nclass MixedWindowDataset(Dataset):\n    \"\"\"\n    Concatenates a normal dataset and an anomaly dataset and exposes a\n    WeightedRandomSampler that enforces the desired ratio.\n\n    Usage:\n        dataset = MixedWindowDataset(normal_ds, anomaly_ds, ratio=9)\n        loader  = dataset.build_loader(batch_size=128)\n    \"\"\"\n\n    def __init__(\n        self,\n        normal_dataset: Dataset,\n        anomaly_dataset: Dataset,\n        ratio: int = 9,          # normal : anomaly  (e.g. 9 means 9:1)\n    ) -> None:\n        self.normal  = normal_dataset\n        self.anomaly = anomaly_dataset\n        self.ratio   = ratio\n\n        n_normal  = len(normal_dataset)\n        n_anomaly = len(anomaly_dataset)\n\n        if n_anomaly == 0:\n            raise ValueError(\"Anomaly dataset is empty \u2013 cannot create mixed loader\")\n\n        # Weight each sample so expected counts match the requested ratio\n        # w_normal : w_anomaly  =  ratio : 1\n        # \u2192 normalise so they sum to 1 within each class\n        w_normal  = ratio  / n_normal\n        w_anomaly = 1.0    / n_anomaly\n        self._weights = (\n            [w_normal]  * n_normal +\n            [w_anomaly] * n_anomaly\n        )\n        self._weights = torch.tensor(self._weights, dtype=torch.float32)\n\n        self._n_total  = n_normal + n_anomaly\n\n    def __len__(self) -> int:\n        return self._n_total\n\n    def __getitem__(self, idx: int) -> torch.Tensor:\n        n = len(self.normal)\n        item = self.normal[idx] if idx < n else self.anomaly[idx - n]\n        # Strip label if the underlying dataset returns (tensor, label) tuples\n        return item[0] if isinstance(item, (list, tuple)) else item\n\n    def build_loader(\n        self,\n        batch_size: int,\n        num_workers: int = 0,\n        num_samples: int | None = None,\n    ) -> DataLoader:\n        \"\"\"\n        Returns a DataLoader that samples with the mix ratio.\n\n        Args:\n            num_samples: total draws per epoch (default: len(normal) * (1 + 1/ratio))\n        \"\"\"\n        if num_samples is None:\n            # One full pass over normal data + proportional anomaly samples\n            num_samples = int(len(self.normal) * (1 + 1 / self.ratio))\n\n        sampler = WeightedRandomSampler(\n            weights     = self._weights,\n            num_samples = num_samples,\n            replacement = True,\n        )\n        return DataLoader(\n            self,\n            batch_size  = batch_size,\n            sampler     = sampler,\n            num_workers = num_workers,\n            pin_memory  = False,   # MPS doesn't benefit from pin_memory\n            drop_last   = True,\n        )\n",
    "evaluation/__init__.py": "",
    "evaluation/hprs.py": "\"\"\"\nHPRS (Harmonic Mean of Precision, Recall, and Specificity) threshold calibration.\n\nDefinition\n----------\nFor a given threshold \u03c4 on the anomaly score:\n  - Predictions: \u0177 = 1 if score(x) \u2265 \u03c4 else 0  (1 = anomalous)\n  - Precision   P  = TP / (TP + FP)\n  - Recall      R  = TP / (TP + FN)   (sensitivity)\n  - Specificity S  = TN / (TN + FP)\n\n  HPRS(\u03c4) = 3 / (1/P + 1/R + 1/S)     [harmonic mean of P, R, S]\n\nHPRS penalises both false positives (via P and S) and false negatives (via R),\nmaking it particularly suitable for shoplifting detection where false alarms\n(phone-use, crouching, bag adjustment) erode operator trust.\n\nUsage\n-----\n    from evaluation.hprs import calibrate_threshold, evaluate\n\n    tau, hprs, stats = calibrate_threshold(scores, labels)\n    print(f\"\u03c4* = {tau:.4f},  HPRS = {hprs:.4f}\")\n    metrics = evaluate(scores, labels, tau)\n\"\"\"\nfrom __future__ import annotations\n\nfrom typing import Dict, Tuple\n\nimport numpy as np\n\n\n_EPS = 1e-8\n\n\ndef _confusion(\n    scores: np.ndarray,\n    labels: np.ndarray,\n    tau: float,\n) -> Tuple[int, int, int, int]:\n    \"\"\"Return (TP, FP, TN, FN) for predictions score \u2265 \u03c4 \u2192 anomalous.\"\"\"\n    preds = (scores >= tau).astype(int)\n    TP = int(((preds == 1) & (labels == 1)).sum())\n    FP = int(((preds == 1) & (labels == 0)).sum())\n    TN = int(((preds == 0) & (labels == 0)).sum())\n    FN = int(((preds == 0) & (labels == 1)).sum())\n    return TP, FP, TN, FN\n\n\ndef hprs_at_tau(\n    scores: np.ndarray,\n    labels: np.ndarray,\n    tau: float,\n) -> float:\n    \"\"\"Compute HPRS at a single threshold.\"\"\"\n    TP, FP, TN, FN = _confusion(scores, labels, tau)\n\n    P = TP / (TP + FP + _EPS)\n    R = TP / (TP + FN + _EPS)\n    S = TN / (TN + FP + _EPS)\n\n    denom = (1 / (P + _EPS)) + (1 / (R + _EPS)) + (1 / (S + _EPS))\n    return 3 / (denom + _EPS)\n\n\ndef calibrate_threshold(\n    scores: np.ndarray,\n    labels: np.ndarray,\n    n_thresholds: int = 1000,\n) -> Tuple[float, float, Dict]:\n    \"\"\"\n    Sweep \u03c4 over the score range and find the value that maximises HPRS.\n\n    Args:\n        scores:       (N,) anomaly scores (higher = more suspicious)\n        labels:       (N,) binary ground-truth labels (1 = shoplifting)\n        n_thresholds: resolution of the sweep\n\n    Returns:\n        tau_star:  optimal threshold\n        hprs_star: HPRS value at tau_star\n        stats:     dict of metrics at tau_star\n    \"\"\"\n    scores = np.asarray(scores, dtype=float)\n    labels = np.asarray(labels, dtype=int)\n\n    lo, hi = float(scores.min()), float(scores.max())\n    taus   = np.linspace(lo, hi, n_thresholds)\n\n    best_tau  = lo\n    best_hprs = -1.0\n\n    for tau in taus:\n        h = hprs_at_tau(scores, labels, tau)\n        if h > best_hprs:\n            best_hprs = h\n            best_tau  = tau\n\n    stats = evaluate(scores, labels, best_tau)\n    return float(best_tau), float(best_hprs), stats\n\n\ndef evaluate(\n    scores: np.ndarray,\n    labels: np.ndarray,\n    tau: float,\n) -> Dict[str, float]:\n    \"\"\"\n    Full evaluation metrics at a fixed threshold.\n\n    Returns dict with:  TP, FP, TN, FN, precision, recall, specificity,\n                        f1, hprs, accuracy, auc_roc (if sklearn available)\n    \"\"\"\n    scores = np.asarray(scores, dtype=float)\n    labels = np.asarray(labels, dtype=int)\n\n    TP, FP, TN, FN = _confusion(scores, labels, tau)\n    P  = TP / (TP + FP + _EPS)\n    R  = TP / (TP + FN + _EPS)\n    S  = TN / (TN + FP + _EPS)\n    F1 = 2 * P * R / (P + R + _EPS)\n    denom = (1 / (P + _EPS)) + (1 / (R + _EPS)) + (1 / (S + _EPS))\n    HPRS  = 3 / (denom + _EPS)\n    ACC   = (TP + TN) / (TP + FP + TN + FN + _EPS)\n\n    metrics: Dict[str, float] = {\n        \"tau\":         tau,\n        \"TP\":          TP,\n        \"FP\":          FP,\n        \"TN\":          TN,\n        \"FN\":          FN,\n        \"precision\":   P,\n        \"recall\":      R,\n        \"specificity\": S,\n        \"f1\":          F1,\n        \"hprs\":        HPRS,\n        \"accuracy\":    ACC,\n    }\n\n    # Optional AUC-ROC\n    try:\n        from sklearn.metrics import roc_auc_score\n        if len(np.unique(labels)) == 2:\n            metrics[\"auc_roc\"] = float(roc_auc_score(labels, scores))\n    except ImportError:\n        pass\n\n    return metrics\n\n\ndef print_report(metrics: Dict[str, float]) -> None:\n    \"\"\"Pretty-print evaluation results.\"\"\"\n    print(\"\u2500\" * 50)\n    print(f\"  Threshold   (\u03c4)  : {metrics['tau']:.6f}\")\n    print(f\"  Precision       : {metrics['precision']:.4f}\")\n    print(f\"  Recall          : {metrics['recall']:.4f}\")\n    print(f\"  Specificity     : {metrics['specificity']:.4f}\")\n    print(f\"  F1              : {metrics['f1']:.4f}\")\n    print(f\"  HPRS            : {metrics['hprs']:.4f}\")\n    print(f\"  Accuracy        : {metrics['accuracy']:.4f}\")\n    if \"auc_roc\" in metrics:\n        print(f\"  AUC-ROC         : {metrics['auc_roc']:.4f}\")\n    print(f\"  TP={int(metrics['TP'])}  FP={int(metrics['FP'])}  \"\n          f\"TN={int(metrics['TN'])}  FN={int(metrics['FN'])}\")\n    print(\"\u2500\" * 50)\n",
}
for fname, content in files.items():
    p = ROOT / fname
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    print(f'  wrote {fname}')
print('Source files ready.')


In [ ]:
# ── Cell 5: Extract RetailS dataset ──────────────────────────────────────────
import zipfile
from pathlib import Path

ROOT     = Path('/content/STG_NF')
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

print('Extracting RetailS.zip (may take a minute) ...')
with zipfile.ZipFile(RETAIL_ZIP) as zf:
    zf.extractall(DATA_DIR)

retail_root = DATA_DIR / 'RetailS'
if not retail_root.exists():
    retail_root = DATA_DIR

train_dir = retail_root / 'RetailS_train' / 'pose' / 'train'
n_clips   = len(list(train_dir.glob('*.json')))
print(f'Train clips: {n_clips}')
assert n_clips > 0, f'No clips found at {train_dir}'
print('Dataset ready.')


In [ ]:
# ── Cell 6: Apply CUDA configuration ─────────────────────────────────────────
import sys, torch
from pathlib import Path

ROOT = Path('/content/STG_NF')
sys.path.insert(0, str(ROOT))
import config

config.DATA_ROOT      = ROOT / 'data'
config.CHECKPOINT_DIR = ROOT / 'checkpoints'
config.BUFFER_DIR     = ROOT / 'buffer'
config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
config.BUFFER_DIR.mkdir(parents=True, exist_ok=True)

# ── Tune here if you want to override auto-detection ──────────────────────────
config.FLOW_DEPTH      = FLOW_DEPTH   # set by Cell 1
config.HIDDEN_CHANNELS = 64
config.BATCH_SIZE      = BATCH_SIZE   # set by Cell 1
config.MAX_EPOCHS      = 60
config.LR              = 1e-4
# ─────────────────────────────────────────────────────────────────────────────

torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device         : {DEVICE}')
print(f'flow_depth     : {config.FLOW_DEPTH}')
print(f'hidden_channels: {config.HIDDEN_CHANNELS}')
print(f'batch_size     : {config.BATCH_SIZE}')
print(f'max_epochs     : {config.MAX_EPOCHS}')


In [ ]:
# ── Cell 7: GPU throughput benchmark ─────────────────────────────────────────
import time, torch
from model.stg_nf import build_model

bench = build_model(config.FLOW_DEPTH, config.HIDDEN_CHANNELS, device=DEVICE)
bench.train()
opt = torch.optim.AdamW(bench.parameters(), lr=1e-4)

for bs in [256, config.BATCH_SIZE]:
    x = torch.randn(bs, 2, 24, 15, device=DEVICE)
    for _ in range(3):
        loss = bench.nll_loss(x); opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(20):
        loss = bench.nll_loss(x); opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()
    its = 20 / (time.time() - t0)
    sps = int(bs * its)
    epoch_min = 1_378_900 / sps / 60
    print(f'batch={bs:5d}  {its:6.1f} it/s  {sps:8,} samples/s  '
          f'~{epoch_min:.1f} min/epoch  '
          f'~{epoch_min * config.MAX_EPOCHS / 60:.1f}h total')

del bench, opt; torch.cuda.empty_cache()


In [ ]:
# ── Cell 8: CUDA-optimised Trainer ───────────────────────────────────────────
import torch, shutil
from pathlib import Path
from tqdm import tqdm
from training.trainer import Trainer
from config import GRAD_CLIP


class CUDATrainer(Trainer):
    """
    Extends Trainer with:
      - bfloat16 Automatic Mixed Precision  (+30-40% throughput)
      - non_blocking GPU transfers
      - best-checkpoint mirrored to Drive after every improvement
    """

    def __init__(self, *args, use_amp=True, drive_ckpt_dir=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.use_amp        = use_amp and self.device.type == 'cuda'
        self.scaler         = torch.cuda.amp.GradScaler() if self.use_amp else None
        self.drive_ckpt_dir = drive_ckpt_dir
        print(f'[CUDATrainer] AMP={self.use_amp}  drive_backup={drive_ckpt_dir is not None}')

    def _epoch(self, loader, training):
        if loader is None:
            return float('nan')
        self.model.train(training)
        total, n = 0.0, 0
        ctx = torch.enable_grad() if training else torch.no_grad()
        with ctx:
            for batch in tqdm(loader, desc='train' if training else 'val ', leave=False):
                x = batch[0] if isinstance(batch, (list, tuple)) else batch
                x = x.to(self.device, non_blocking=True)
                if training and self.use_amp:
                    with torch.autocast('cuda', dtype=torch.bfloat16):
                        loss = self.model.nll_loss(x)
                    self.optimizer.zero_grad(set_to_none=True)
                    self.scaler.scale(loss).backward()
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    loss = self.model.nll_loss(x)
                    if training:
                        self.optimizer.zero_grad(set_to_none=True)
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                        self.optimizer.step()
                total += float(loss.detach()); n += 1
        return total / max(n, 1)

    def _save(self, name, epoch):
        super()._save(name, epoch)
        if self.drive_ckpt_dir and name == 'best.pt':
            dst = Path(self.drive_ckpt_dir)
            dst.mkdir(parents=True, exist_ok=True)
            src = self.ckpt_dir / f'{self.run_name}_best.pt'
            shutil.copy2(src, dst / f'{self.run_name}_best.pt')
            print(f'  [Drive] Backed up best.pt')


print('CUDATrainer ready.')


In [ ]:
# ── Cell 9: Load data ─────────────────────────────────────────────────────────
from torch.utils.data import DataLoader, random_split
from dataset.retail_dataset import RetailSTrainDataset, RetailSTestDataset
from training.mixing import MixedWindowDataset

retail_root = config.DATA_ROOT / 'RetailS'
WORKERS     = 4  # safe on Colab Linux

print('Loading normal training windows (~2 min) ...')
full_ds  = RetailSTrainDataset(retail_root)
val_size = int(len(full_ds) * 0.05)
train_ds, val_ds = random_split(full_ds, [len(full_ds) - val_size, val_size])

print('Building 9:1 mixed dataset ...')
anomaly_ds   = RetailSTestDataset(retail_root, split='staged', anomaly_only=True)
mixed_ds     = MixedWindowDataset(train_ds, anomaly_ds, ratio=9)

train_loader = mixed_ds.build_loader(
    batch_size=config.BATCH_SIZE, num_workers=WORKERS)
val_loader   = DataLoader(val_ds, batch_size=config.BATCH_SIZE * 2,
                          num_workers=WORKERS, pin_memory=True, shuffle=False)

print(f'train={len(mixed_ds):,}  val={len(val_ds):,}  workers={WORKERS}')


In [ ]:
# ── Cell 10: Build + compile model ───────────────────────────────────────────
import torch
from model.stg_nf import build_model

model = build_model(config.FLOW_DEPTH, config.HIDDEN_CHANNELS, device=DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {params:,}')

if hasattr(torch, 'compile'):
    print('Compiling with torch.compile (first batch will be slow) ...')
    model = torch.compile(model)
    print('Done.')


In [ ]:
# ── Cell 11: Train ────────────────────────────────────────────────────────────
from pathlib import Path

DRIVE_CKPT  = f'{DRIVE_ROOT}/STG_NF_checkpoints'
resume_path = config.CHECKPOINT_DIR / 'stgnf_best.pt'

if not resume_path.exists():
    drive_best = Path(f'{DRIVE_CKPT}/stgnf_best.pt')
    if drive_best.exists():
        import shutil
        shutil.copy2(drive_best, resume_path)
        print(f'Restored from Drive: {drive_best}')
    else:
        resume_path = None

trainer = CUDATrainer(
    model          = model,
    device         = DEVICE,
    lr             = config.LR,
    max_epochs     = config.MAX_EPOCHS,
    ckpt_dir       = config.CHECKPOINT_DIR,
    run_name       = 'stgnf',
    use_amp        = True,
    drive_ckpt_dir = DRIVE_CKPT,
)

print(f'Training  epochs={config.MAX_EPOCHS}  device={DEVICE}')
trainer.fit(train_loader=train_loader, val_loader=val_loader, resume_from=resume_path)


In [ ]:
# ── Cell 12: Calibrate HPRS threshold ────────────────────────────────────────
import json, numpy as np
from torch.utils.data import DataLoader
from training.trainer import load_checkpoint
from dataset.retail_dataset import RetailSTestDataset
from evaluation.hprs import calibrate_threshold, print_report

best_path = config.CHECKPOINT_DIR / 'stgnf_best.pt'
load_checkpoint(model, best_path, device=DEVICE)
model.eval()

test_ds = RetailSTestDataset(retail_root, split='staged')
loader  = DataLoader(test_ds, batch_size=512, num_workers=WORKERS, pin_memory=True)

scores, labels = [], []
import torch
with torch.no_grad():
    for x, lbl in loader:
        scores.extend(model.anomaly_score(x.to(DEVICE)).cpu().tolist())
        labels.extend(lbl.tolist())

tau, hprs, stats = calibrate_threshold(np.array(scores), np.array(labels))
print_report(stats)

with open(config.CHECKPOINT_DIR / 'tau.json', 'w') as f:
    json.dump({'tau': tau, 'hprs': hprs, **{k: float(v) for k, v in stats.items()}}, f, indent=2)
print(f'Saved tau={tau:.4f}  HPRS={hprs:.4f}')


In [ ]:
# ── Cell 13: Loss curve ───────────────────────────────────────────────────────
import json, matplotlib.pyplot as plt

with open(config.CHECKPOINT_DIR / 'stgnf_metrics.json') as f:
    metrics = json.load(f)

epochs    = [m['epoch']     for m in metrics]
train_nll = [m['train_nll'] for m in metrics]
val_nll   = [m['val_nll']   for m in metrics]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, train_nll, label='Train NLL', color='steelblue')
axes[0].plot(epochs, val_nll,   label='Val NLL',   color='tomato', linestyle='--')
axes[0].set(xlabel='Epoch', ylabel='NLL', title='Training Curve')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [m['lr'] for m in metrics], color='green')
axes[1].set(xlabel='Epoch', ylabel='LR', title='Learning Rate')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(config.CHECKPOINT_DIR / 'training_curve.png', dpi=150)
plt.show()


In [ ]:
# ── Cell 14: Save all artefacts to Drive ─────────────────────────────────────
import shutil
from pathlib import Path

dst = Path(DRIVE_CKPT)
dst.mkdir(parents=True, exist_ok=True)

for fname in ['stgnf_best.pt', 'stgnf_last.pt', 'tau.json',
               'stgnf_metrics.json', 'training_curve.png']:
    src = config.CHECKPOINT_DIR / fname
    if src.exists():
        shutil.copy2(src, dst / fname)
        print(f'  {fname} -> Drive')

print(f'All artefacts saved to: {DRIVE_CKPT}')


## Running trained weights locally (M4 MacBook)

1. Download `stgnf_best.pt` and `tau.json` from `My Drive/STG_NF_checkpoints/`
2. Place them in `script/STG_NF/checkpoints/`
3. Run the live detector:

```bash
cd script/STG_NF
source .venv/bin/activate
python main_detect.py --weights checkpoints/stgnf_best.pt
```

`tau.json` is loaded automatically. The detector runs on MPS (your M4 GPU) at 5 FPS.
